# Drawing Networks: How Network Layout Algorithms Work and How Much to Trust Them
In this chapter we will:

1. Treat a network drawing as the output of an algorithm, not as a picture of the network
2. Build the Fruchterman–Reingold layout (the algorithm behind `nx.spring_layout`) from scratch, and compare it with other energy models
3. Measure how stable and how faithful a layout is, and design drawings whose positions mean something on purpose

*Running example: the Venice Biennale network (ASAC archive + Artfacts/Artnet data). The Biennale is the case study, not the topic: we use it to build progressively more defensible visualizations.*

*Acknowledgement: This chapter relies on the force-directed layout literature (Eades; Fruchterman & Reingold; Kamada & Kawai; Jacomy et al.) and on the visual network analysis framework of Venturini, Jacomy & Jensen (2021).*
__________

## Plan
*For the intermediate presentations on Monday, Oct 19.*

### The problem

**Seeing before reading: Gestalt principles in network drawings.** Every network figure is a visual claim, often perceived before textual or numerical information. Gestalt psychology described how the eye groups what it sees early in the twentieth century (Wertheimer, 1923). A century of experiments has refined and confirmed those rules (Wagemans et al., 2012). In a node-link diagram they work like this:

- **Proximity:** nodes drawn close together are seen as similar or related.
- **Connectedness:** nodes joined by lines are seen as a unit. This grouping is even stronger than proximity (Palmer & Rock, 1994).
- **Common region:** dense clumps of nodes are seen as groups, that is, as communities.
- **Similarity:** nodes with the same colour or shape are seen as belonging together. Differences in colour "pop out" before conscious attention (Healey & Enns, 2012).
- **Centre and size:** what sits in the middle of the frame, or is drawn biggest, is read as most important. Size is also judged poorly: readers estimate area much less accurately than position or length (Cleveland & McGill, 1984).

These effects are not abstract. Experiments with social network drawings show that the same network, drawn with different layouts, leads people to different judgements about who is central and which groups exist (McGrath, Blythe & Krackhardt, 1997).

**The positions are not measurements.** Yet in a force-directed drawing, none of those positions is measured. Each one is the output of an optimisation: a local minimum of an energy function, reached from a random start. Change the seed and the "map" moves. Change the energy and the "communities" change (Noack, 2009). The reader's eye applies Gestalt rules to coordinates that encode partly the network and partly the algorithm's choices and its randomness. Venturini, Jacomy & Jensen (2021) describe this gap between what a drawing suggests and what the data supports as a problem of *relational ambiguity*.

**Drawings as evidence in visual science communication.** The gap matters because scientific images do not just illustrate results. They persuade, and they travel further than the analysis behind them (Latour, 1986). A network figure in a paper, a press release or a slide is often the only part of a study most readers ever see. Small design choices can change what viewers conclude from a chart (Pandey et al., 2015). A node-link diagram involves many more such choices than a bar chart: the layout algorithm, its parameters, the seed, and the encodings of size and colour. A layout that looks authoritative but is unstable becomes a claim no one actually tested.

**How this extends existing practice.** In practice the layout is almost never questioned. Almost every notebook in this course, and many network figures in published papers, begin with `nx.spring_layout` and its default parameters. Often the picture is read before the structure is measured, or instead of measuring it.

The tools for questioning it already exist, in two places:

- graph-drawing research offers quality metrics;
- visual network analysis offers a critique of how layouts are read.

The two are rarely turned into a routine check that analysts run on their own figures. This chapter proposes that routine in three steps: build the algorithm, measure its output, then design the drawing on purpose.

**The case study: inclusion and exclusion at the Venice Biennale.** The stakes are highest when a network is about inclusion and exclusion. In a network of the origins of artists who exhibited at the Venice Biennale, the question is literally who sits at the centre and who at the margin. This is not a neutral question in the art world: success depends strongly on early access to a core of prestigious institutions (Fraiberger et al., 2018); the globalisation of contemporary art remains uneven, with artists from a small set of countries still dominating international exhibitions (Quemin, 2006).

A force-directed layout tends to push low-degree nodes to the edge and to exaggerate gaps between groups, partly because of the random start and the force laws themselves. Suppose a figure places Global South artists at the periphery. If part of that placement comes from the algorithm rather than from how the artists are actually connected, the figure carries a visual bias, and misreading it means misreading the finding. For curators, funders and cultural policy makers who rely on such figures, the cost is concrete. It shapes which artists and regions are seen as part of the conversation. Who is drawn, and where, is also a question of who gets counted (D'Ignazio & Klein, 2020).

### Where this chapter fits in the book

The book already covers two things: how to *compute* structure (centralities in Class 04, communities in Classes 07–08) and how to *draw* a network in Python (Visualization, Class 11). **The step in between is missing: what the drawing algorithm does with the structure, and whether the picture agrees with the measurements.**

This chapter fills that gap using tools readers already have:

- degree-preserving rewiring from Class 04 becomes a null model for layouts;
- modularity from Classes 07–08 reappears as an energy function that a layout minimises (Noack, 2009).

**What the chapter contributes.** The algorithms and the critique of how layouts are read live in separate literatures:

- the algorithms come from graph drawing in computer science (Eades, 1984; Fruchterman & Reingold, 1991; Jacomy et al., 2014);
- the critique comes from visual network analysis in digital methods (Venturini, Jacomy & Jensen, 2021).

Teaching material rarely puts the two side by side. The chapter joins them in one runnable lesson in three moves:

1. build the algorithm from scratch;
2. test its output with network-science tools: seed stability, fidelity metrics, null models;
3. design layouts whose positions encode something on purpose.

### Why one algorithm: Fruchterman–Reingold

The chapter is built around a single algorithm, Fruchterman–Reingold (FR), for three reasons:

1. **It is the one we already use.** `nx.spring_layout` implements FR, so opening it up explains the default behind every network figure in the course.
2. **It is small enough to write from scratch.** It needs two force laws, a temperature and a loop: about 30 lines of NumPy.
3. **It is a template.** Most force-directed methods (LinLog, ForceAtlas2) keep FR’s loop and change only the force laws. Once FR is built, Section 3 compares other models by editing one line of our own code, not by importing black boxes.

Kamada–Kawai is the one contrast we keep, because it optimises a different kind of objective: stress on graph distances rather than forces.

### Table of contents

#### 1. What is a graph layout?

**Question:** What problem is a layout algorithm solving, and what makes one drawing better than another?

1. The graph-drawing problem: placing nodes in 2D has no unique answer, so every layout is a choice.
2. A short lineage. It runs from hand-drawn sociograms (Moreno), to Tutte’s barycentric embedding (fix a boundary, place every other node at its neighbours’ average), to spring embedders (Eades).
3. Aesthetic criteria (edge crossings, uniform edge length, symmetry) and the experimental evidence on which ones actually help readers (Purchase).
4. The running example: the Biennale network drawn with defaults. This is the hairball we will improve.

> 📚 Moreno (1934); Tutte (1963); Eades (1984); Purchase (1997); Di Battista et al. (1999)

**→ Next:** Tutte and Eades already contain the key idea: edges pull nodes together, everything else pushes them apart. Section 2 turns that idea into code.

#### 2. Fruchterman–Reingold from scratch *(core section)*

**Question:** What does `nx.spring_layout` actually compute, step by step?

1. **Intuition** (Eades): edges are springs, and all nodes repel each other.
2. **Force laws:** attraction $f_a(d) = d^2/k$ along edges and repulsion $f_r(d) = k^2/d$ between all pairs, where $k$ is the ideal edge length.
3. **One iteration:** sum the forces into a displacement per node, cap it by the temperature $t$, then update the positions inside the frame.
4. **Cooling:** why $t$ must decrease, and what goes wrong when it cools too fast or too slowly.
5. **Validation:** on toy graphs (path, grid, karate club), compare our output with `nx.spring_layout`, both visually and by stress.
6. **Run on the Biennale network.**

> 📚 Eades (1984); Fruchterman & Reingold (1991); Kobourov (2012)

*🔭 Going further: repulsion between all pairs costs O(n²) per iteration. Barnes–Hut quadtrees bring it down to O(n log n) (Barnes & Hut, 1986), and multilevel methods handle large graphs (Hu, 2005).*

**→ Next:** our FR is one choice of force laws. What happens to the picture if we change them?

#### 3. Same graph, different energies

**Question:** How much of the picture comes from the force laws we chose rather than from the network?

1. Rewrite FR as an energy to minimise: each force law is the derivative of an energy term.
2. Swap the force laws inside our own implementation:
    - LinLog (linear attraction energy, logarithmic repulsion energy);
    - ForceAtlas2-style degree-weighted repulsion.
3. The contrast case: Kamada–Kawai minimises the stress between drawn distances and shortest-path distances (`nx.kamada_kawai_layout`).
4. The bridge to community detection: minima of LinLog-type energies are related to modularity (Noack). This links back to Classes 07–08.
5. Draw the same Biennale network with four layouts and compare the “clusters” each one shows.

> 📚 Kamada & Kawai (1989); Noack (2009); Jacomy et al. (2014)

*🔭 Going further: stress majorization, a more robust way to optimise Kamada–Kawai’s objective (Gansner, Koren & North, 2005).*

**→ Next:** if different, reasonable choices give different pictures, we need ways to measure what a single picture can be trusted for.

#### 4. How much should we trust a layout? *(critical section)*

**Question:** Which features of a drawing are robust properties of the network, and which are artifacts of the algorithm?

1. **Stochasticity:** run FR over many seeds, align the results with Procrustes, and map each node’s positional uncertainty.
2. **Fidelity:** compute stress and neighbourhood preservation. Does drawn distance track graph distance?
3. **Null-model check:** lay out degree-preserving rewired graphs, using the double-edge swap from Class 04. Any structure that survives the rewiring comes from the degrees, not from the data.
4. **The centrality illusion:** correlate each node’s distance from the centre of the drawing with its degree and betweenness.

> 📚 Venturini, Jacomy & Jensen (2021); Kruiger et al. (2017); Purchase (1997)

*🔭 Going further: edge crossings as an extra quality metric. Layouts that optimise neighbourhood preservation directly, such as t-SNE-based graph layouts (Kruiger et al., 2017).*

**→ Next:** once we know what a free layout cannot guarantee, we can stop leaving meaning to chance and assign it ourselves.

#### 5. From picture to argument: layouts that encode on purpose

**Question:** How can we design drawings in which position means something we chose and can defend?

1. **Radial layout constrained by k-core or centrality:** distance from the centre becomes a measured value.
2. **Attribute-pinned positions:** fix one coordinate by an attribute (e.g. Global North / Global South) and let FR place the other.
3. **Anchored small multiples across Biennale editions,** to preserve the mental map over time.
4. **When node-link stops working:** an adjacency matrix for dense graphs.
5. **Encoding check:** node size as area vs. as radius.

> 📚 Brandes, Kenis & Wagner (2003); Misue et al. (1995); Ghoniem, Fekete & Castagliola (2005); Cleveland & McGill (1984)

*🔭 Going further: edge bundling for dense drawings (Holten & van Wijk, 2009). Interactive exploration with ipysigma.*

**→ Next:** the reader now has tools both to test a layout and to design one. The exercise asks them to use both.

#### 6. Your Turn and Limitations

**Question:** Can the reader change the algorithm and measure the consequences on their own?

1. **Exercise:** change the cooling schedule (or the repulsion law) of our FR, then evaluate the result with the Section 4 metrics.
2. **Limitations:**
    - a 2D drawing cannot preserve high-dimensional structure;
    - O(n²) methods do not scale;
    - quality metrics do not capture everything readers perceive.

> 📚 Kobourov (2012); Venturini, Jacomy & Jensen (2021)

---

### Data

The running example uses Venice Biennale Arte artists, exhibitions and national pavilions. The sources are the ASAC archive plus Artfacts/Artnet exhibition histories, collected for my master’s thesis; on networks in the art world, see Fraiberger et al. (2018). A cleaned subset goes in `final_project/data/` so the notebook runs top to bottom. Toy graphs (path, grid, karate club) come first.

### Work plan

| Weeks | Goal | Deliverable |
| --- | --- | --- |
| Oct 19 – Nov 1 | Sections 1–2 + data | From-scratch FR checked against `nx.spring_layout`; Biennale subset cleaned |
| Nov 2 – Nov 15 | Section 3 | LinLog and ForceAtlas2 force laws in our FR; Kamada–Kawai; side-by-side figures |
| Nov 16 – Nov 29 | Section 4 | Seed stability, fidelity metrics, null model, centrality illusion; check-in with Brennan |
| Nov 30 – Dec 9 | Section 5 + polish | Constrained layouts, Your Turn, Limitations, references; restart & run all |
| Dec 14 / 16 | Final presentation | Finished chapter |

__________

In [ ]:
# Import your packages! Everything below is in the course environment (cmns-core).
# Optional, only for the "Going further" parts: pip install ipysigma
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import procrustes   # aligning layouts across seeds (Section 4)
from matplotlib import rc
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')
rc('axes', axisbelow=True)

### Motivation: which picture is true?
Below is the same network, drawn by the same function we use in every class, with two different random seeds. The two red nodes are the same in both panels. Keep this figure in mind: the rest of the chapter explains why the pictures differ, and which features of either one we can trust.

In [ ]:
# Same network, same algorithm, two random seeds
G = nx.karate_club_graph()
colors = ['crimson' if n in (0, 33) else '#4C72B0' for n in G]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=120)
for ax, seed in zip(axes, [1, 2]):
    pos = nx.spring_layout(G, seed=seed)
    nx.draw_networkx(G, pos, ax=ax, with_labels=False, node_color=colors,
                     node_size=[25 * G.degree(n) for n in G], edge_color='.75')
    ax.set_title(f"nx.spring_layout, seed = {seed}", fontsize=12)
    ax.axis('off')
fig.suptitle("Same network, same algorithm: which picture is true?", fontsize=14)
plt.show()

## 1. What is a graph layout?
**Question:** What problem is a layout algorithm solving, and what makes one drawing better than another?

*To be written.*
________

## 2. Fruchterman–Reingold from scratch (core section)
**Question:** What does `nx.spring_layout` actually compute, step by step?

*To be written.*
________

In [ ]:
def fruchterman_reingold(G, iterations=50, seed=None):
    """
    Force-directed layout following Fruchterman & Reingold (1991),
    implemented from scratch with NumPy.


    Parameters
    ----------
    G : networkx.Graph
        Undirected graph to lay out.
    iterations : int
        Number of force-update steps; the temperature cools to zero.
    seed : int or None
        Seed for the random initial positions.


    Returns
    -------
    pos : dict
        Node -> np.array([x, y]).


    Notes
    -----
    - Attraction f_a(d) = d**2 / k along edges; repulsion f_r(d) = k**2 / d
      between all pairs, with k the ideal edge length.
    - Each step's displacement is capped by the current temperature.


    Time Complexity
    ---------------
    O(iterations * n**2), because repulsion is computed for all pairs.


    Citations
    ---------
    Fruchterman, T. M. J., & Reingold, E. M. (1991). Graph drawing by
    force-directed placement. Software: Practice and Experience, 21(11),
    1129-1164. doi: https://doi.org/10.1002/spe.4380211102
    """
    # TODO (Section 2): random start, then loop over forces and cooling
    pass

## 3. Same graph, different energies
**Question:** How much of the picture comes from the force laws we chose rather than from the network?

*To be written.*
________

## 4. How much should we trust a layout? (critical section)
**Question:** Which features of a drawing are robust properties of the network, and which are artifacts of the algorithm?

*To be written.*
________

## 5. From picture to argument: layouts that encode on purpose
**Question:** How can we design drawings in which position means something we chose and can defend?

*To be written.*
________

### Your turn!
Change the cooling schedule of `fruchterman_reingold` (for example exponential instead of linear) or its repulsion law. Then use the Section 4 metrics to measure whether the layout becomes more stable, or more faithful to graph distances.

In [ ]:
# Your Turn!

## Limitations
*To be written.* Where layouts break, and what they can't tell us:
- a 2D drawing cannot preserve high-dimensional structure;
- O(n²) methods do not scale;
- quality metrics do not capture everything readers perceive.

_______

## Annotated bibliography
Importance: ⭐⭐⭐ essential (start here) · ⭐⭐ important · ⭐ background. *Used in* = chapter sections.

### 1. Perception: how readers see a network drawing
- ⭐⭐ **Cleveland, W. S., & McGill, R. (1984).** Graphical perception: Theory, experimentation, and application to the development of graphical methods. *JASA*, 79(387), 531–554. [doi:10.1080/01621459.1984.10478080](https://doi.org/10.1080/01621459.1984.10478080)
  *Ranking of visual encodings: position > length > area.* · Used in: 5, Intro
- ⭐⭐ **Healey, C. G., & Enns, J. T. (2012).** Attention and visual memory in visualization and computer graphics. *IEEE TVCG*, 18(7), 1170–1188. [doi:10.1109/TVCG.2011.127](https://doi.org/10.1109/TVCG.2011.127)
  *Preattentive features: colour, size, orientation "pop out".* · Used in: 5, Intro
- ⭐⭐ **Palmer, S., & Rock, I. (1994).** Rethinking perceptual organization: The role of uniform connectedness. *Psychonomic Bulletin & Review*, 1(1), 29–55. [doi:10.3758/BF03200760](https://doi.org/10.3758/BF03200760)
  *Connectedness groups more strongly than proximity — edges dominate.* · Used in: Intro
- ⭐⭐ **Wagemans, J., Elder, J. H., Kubovy, M., Palmer, S. E., Peterson, M. A., Singh, M., & von der Heydt, R. (2012).** A century of Gestalt psychology in visual perception: I. Perceptual grouping and figure–ground organization. *Psychological Bulletin*, 138(6), 1172–1217. [doi:10.1037/a0029333](https://doi.org/10.1037/a0029333)
  *Modern review confirming Gestalt grouping still holds.* · Used in: Intro
- ⭐ **Wertheimer, M. (1923).** Untersuchungen zur Lehre von der Gestalt II. *Psychologische Forschung*, 4, 301–350. [doi:10.1007/BF00410640](https://doi.org/10.1007/BF00410640)
  *Founding text of Gestalt grouping principles (proximity, similarity, continuation).* · Used in: Intro

### 2. Network drawings as evidence
- ⭐⭐⭐ **McGrath, C., Blythe, J., & Krackhardt, D. (1997).** The effect of spatial arrangement on judgments and errors in interpreting graphs. *Social Networks*, 19(3), 223–242. [doi:10.1016/S0378-8733(96)00299-7](https://doi.org/10.1016/S0378-8733(96)00299-7)
  *Same network, different layouts → different judgements of centrality and groups.* · Used in: 4, Intro
- ⭐⭐⭐ **Venturini, T., Jacomy, M., & Jensen, P. (2021).** What do we see when we look at networks: Visual network analysis, relational ambiguity, and force-directed layouts. *Big Data & Society*, 8(1). [doi:10.1177/20539517211018488](https://doi.org/10.1177/20539517211018488)
  *What force-directed layouts can and cannot legitimately show.* · Used in: 4, Intro
- ⭐⭐ **Pandey, A. V., Rall, K., Satterthwaite, M. L., Nov, O., & Bertini, E. (2015).** How deceptive are deceptive visualizations? An empirical analysis of common distortion techniques. *Proc. CHI 2015*, 1469–1478. [doi:10.1145/2702123.2702608](https://doi.org/10.1145/2702123.2702608)
  *Small design distortions change readers' conclusions.* · Used in: Intro
- ⭐ **Latour, B. (1986).** Visualisation and cognition: Drawing things together. *Knowledge and Society*, 6, 1–40.
  *Scientific images (inscriptions) persuade and circulate.* · Used in: Intro

### 3. Foundations of graph drawing
- ⭐⭐⭐ **Kobourov, S. G. (2012).** Spring embedders and force directed graph drawing algorithms. [arXiv:1201.3011](https://arxiv.org/abs/1201.3011)
  *Readable survey of force-directed methods, Eades to multilevel.* · Used in: 1, 2, 3
- ⭐⭐ **Eades, P. (1984).** A heuristic for graph drawing. *Congressus Numerantium*, 42, 149–160. (no online version; summarised in Kobourov 2012)
  *Original spring embedder; start of force-directed layout.* · Used in: 1, 2
- ⭐⭐ **Purchase, H. (1997).** Which aesthetic has the greatest effect on human understanding? *Graph Drawing (GD '97)*, LNCS 1353, 248–261. [doi:10.1007/3-540-63938-1_67](https://doi.org/10.1007/3-540-63938-1_67)
  *User study: fewer edge crossings help comprehension most.* · Used in: 1, 4
- ⭐⭐ **Tutte, W. T. (1963).** How to draw a graph. *Proceedings of the London Mathematical Society*, s3-13(1), 743–767. [doi:10.1112/plms/s3-13.1.743](https://doi.org/10.1112/plms/s3-13.1.743)
  *Barycentric embedding; first algorithmic layout.* · Used in: 1
- ⭐ **Di Battista, G., Eades, P., Tamassia, R., & Tollis, I. G. (1999).** *Graph Drawing: Algorithms for the Visualization of Graphs*. Prentice Hall. (book, library)
  *Standard graph-drawing textbook; definitions and layout families.* · Used in: 1
- ⭐ **Moreno, J. L. (1934).** *Who Shall Survive? A New Approach to the Problem of Human Interrelations*. Nervous and Mental Disease Publishing Co. (book, library)
  *First hand-drawn sociograms; origin of network visualization.* · Used in: 1

### 4. Force-directed algorithms
- ⭐⭐⭐ **Fruchterman, T. M. J., & Reingold, E. M. (1991).** Graph drawing by force-directed placement. *Software: Practice and Experience*, 21(11), 1129–1164. [doi:10.1002/spe.4380211102](https://doi.org/10.1002/spe.4380211102)
  *The FR algorithm behind nx.spring_layout; implemented from scratch.* · Used in: 2
- ⭐⭐⭐ **Jacomy, M., Venturini, T., Heymann, S., & Bastian, M. (2014).** ForceAtlas2, a continuous graph layout algorithm for handy network visualization designed for the Gephi software. *PLoS ONE*, 9(6), e98679. [doi:10.1371/journal.pone.0098679](https://doi.org/10.1371/journal.pone.0098679)
  *Gephi's layout: degree-weighted repulsion, LinLog mode.* · Used in: 3
- ⭐⭐⭐ **Noack, A. (2009).** Modularity clustering is force-directed layout. *Physical Review E*, 79(2), 026102. [doi:10.1103/PhysRevE.79.026102](https://doi.org/10.1103/PhysRevE.79.026102)
  *LinLog layout energy ≈ modularity; layouts ↔ communities.* · Used in: 3
- ⭐⭐ **Kamada, T., & Kawai, S. (1989).** An algorithm for drawing general undirected graphs. *Information Processing Letters*, 31(1), 7–15. [doi:10.1016/0020-0190(89)90102-6](https://doi.org/10.1016/0020-0190(89)90102-6)
  *Stress energy matching drawn vs. shortest-path distances.* · Used in: 3
- ⭐ **Barnes, J., & Hut, P. (1986).** A hierarchical O(N log N) force-calculation algorithm. *Nature*, 324, 446–449. [doi:10.1038/324446a0](https://doi.org/10.1038/324446a0)
  *Quadtree approximation to scale repulsion.* · Used in: 2
- ⭐ **Gansner, E. R., Koren, Y., & North, S. (2005).** Graph drawing by stress majorization. *Graph Drawing (GD 2004)*, LNCS 3383, 239–250. [doi:10.1007/978-3-540-31843-9_25](https://doi.org/10.1007/978-3-540-31843-9_25)
  *Robust optimisation of Kamada–Kawai stress.* · Used in: 3
- ⭐ **Hu, Y. (2005).** Efficient, high-quality force-directed graph drawing. *The Mathematica Journal*, 10(1), 37–71.
  *Multilevel force-directed layout for large graphs.* · Used in: 2

### 5. Evaluating and designing layouts
- ⭐⭐ **Brandes, U., Kenis, P., & Wagner, D. (2003).** Communicating centrality in policy network drawings. *IEEE TVCG*, 9(2), 241–253. [doi:10.1109/TVCG.2003.1196010](https://doi.org/10.1109/TVCG.2003.1196010)
  *Radial layouts where distance from centre = centrality.* · Used in: 5
- ⭐⭐ **Ghoniem, M., Fekete, J.-D., & Castagliola, P. (2005).** On the readability of graphs using node-link and matrix-based representations. *Information Visualization*, 4(2), 114–135. [doi:10.1057/palgrave.ivs.9500092](https://doi.org/10.1057/palgrave.ivs.9500092)
  *Adjacency matrices beat node-link for large/dense graphs.* · Used in: 5
- ⭐⭐ **Kruiger, J. F., Rauber, P. E., Martins, R. M., Kerren, A., Kobourov, S., & Telea, A. C. (2017).** Graph layouts by t-SNE. *Computer Graphics Forum*, 36(3), 283–294. [doi:10.1111/cgf.13187](https://doi.org/10.1111/cgf.13187)
  *Neighbourhood preservation as layout-quality metric.* · Used in: 4
- ⭐⭐ **Misue, K., Eades, P., Lai, W., & Sugiyama, K. (1995).** Layout adjustment and the mental map. *Journal of Visual Languages & Computing*, 6(2), 183–210. [doi:10.1006/jvlc.1995.1010](https://doi.org/10.1006/jvlc.1995.1010)
  *Preserving the reader's mental map across changing layouts.* · Used in: 5
- ⭐ **Holten, D., & van Wijk, J. J. (2009).** Force-directed edge bundling for graph visualization. *Computer Graphics Forum*, 28(3), 983–990. [doi:10.1111/j.1467-8659.2009.01450.x](https://doi.org/10.1111/j.1467-8659.2009.01450.x)
  *Edge bundling to reduce clutter in dense drawings.* · Used in: 5

### 6. Case study: visibility in the art world
- ⭐⭐ **Fraiberger, S. P., Sinatra, R., Resch, M., Riedl, C., & Barabási, A.-L. (2018).** Quantifying reputation and success in art. *Science*, 362(6416), 825–829. [doi:10.1126/science.aau7224](https://doi.org/10.1126/science.aau7224)
  *Exhibition networks: early prestige access shapes art careers.* · Used in: Data, Intro
- ⭐⭐ **Quemin, A. (2006).** Globalization and mixing in the visual arts: An empirical survey of "high culture" and globalization. *International Sociology*, 21(4), 522–550.
  *Global art world still dominated by a few countries.* · Used in: 5, Intro
- ⭐ **D'Ignazio, C., & Klein, L. F. (2020).** *Data Feminism*. MIT Press. [open access](https://data-feminism.mitpress.mit.edu/)
  *Who gets counted and made visible in data.* · Used in: Intro

### 7. Software
- ⭐⭐ **Hagberg, A. A., Schult, D. A., & Swart, P. J. (2008).** Exploring network structure, dynamics, and function using NetworkX. *Proc. 7th Python in Science Conference*, 11–15. [conference.scipy.org](https://conference.scipy.org/proceedings/SciPy2008/paper_2/)
  *NetworkX library: spring_layout and kamada_kawai_layout.* · Used in: 2, 3
